In [0]:
# MLflow setup (Volume-backed tracking)
import os
import mlflow

MLRUNS_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/mlruns"
os.makedirs(MLRUNS_PATH, exist_ok=True)

mlflow.set_tracking_uri(f"file:{MLRUNS_PATH}")
mlflow.set_experiment("day13_model_compare")

print("tracking_uri:", mlflow.get_tracking_uri())

In [0]:
spark.sql("SHOW DATABASES").show(truncate=False)

In [0]:
%sql
SHOW TABLES IN gold;

In [0]:
# Load the training table from day 12
from pyspark.sql import functions as F

df = spark.table("gold.ml_training_product_day")
df.printSchema()
display(df.limit(10))
print("rows:", df.count())

In [0]:
# Auto-detect label + feature columns
# Pick a label column (preferred order)
label_candidates = ["purchases", "purchase_count", "target", "label", "revenue"]
label = next((c for c in label_candidates if c in df.columns), None)
if not label:
    raise Exception(f"No label found. Looked for {label_candidates}. Columns: {df.columns}")

# Numeric columns as features, excluding label and obvious IDs/dates
exclude = {label, "event_date", "date", "day", "product_id", "category_id", "category_code", "brand"}
numeric_types = {"int", "bigint", "double", "float", "long", "decimal", "smallint", "tinyint"}

feature_cols = []
for name, dtype in df.dtypes:
    if name in exclude:
        continue
    if any(t in dtype for t in numeric_types):
        feature_cols.append(name)

if len(feature_cols) < 2:
    raise Exception(f"Not enough numeric feature columns found. Found: {feature_cols}")

print("Label:", label)
print("Features:", feature_cols)


In [0]:
# Basic cleanup + train/test split
from pyspark.sql import functions as F

data = (
    df.select([F.col(c).cast("double").alias(c) for c in feature_cols + [label]])
      .na.fill(0)
      .filter(F.col(label).isNotNull())
)

train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)
print("train:", train_df.count(), "test:", test_df.count())

In [0]:
# Pipelines + evaluators
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

def eval_regression(preds, label_col):
    rmse = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse").evaluate(preds)
    mae  = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae").evaluate(preds)
    r2   = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2").evaluate(preds)
    return {"rmse": float(rmse), "mae": float(mae), "r2": float(r2)}

In [0]:
import mlflow
import mlflow.spark

from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor

models = {
    "spark_lr": LinearRegression(featuresCol="features", labelCol=label),
    "spark_rf": RandomForestRegressor(featuresCol="features", labelCol=label, seed=42),
    "spark_gbt": GBTRegressor(featuresCol="features", labelCol=label, seed=42),
}

results = []

for name, algo in models.items():
    pipe = Pipeline(stages=[assembler, algo])

    with mlflow.start_run(run_name=name):
        mlflow.log_param("model_name", name)
        mlflow.log_param("label", label)
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_param("features", ",".join(feature_cols))

        fitted = pipe.fit(train_df)
        preds = fitted.transform(test_df)

        metrics = eval_regression(preds, label)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        mlflow.spark.log_model(fitted, "model",dfs_tmpdir="/Volumes/workspace/ecommerce/ecommerce_data/mlruns/tmp", pip_requirements=["pyspark==4.0.0", "mlflow"])

        results.append({"model": name, **metrics})
        print(name, metrics)

results


In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/ecommerce/ecommerce_data/mlruns/tmp"

# Hyperparameter tuning (RF + GBT) with TrainValidationSplit
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

def tune(model_name, regressor, param_grid):
    pipe = Pipeline(stages=[assembler, regressor])
    evaluator = RegressionEvaluator(labelCol=label, predictionCol="prediction", metricName="rmse")
    tvs = TrainValidationSplit(
        estimator=pipe,
        estimatorParamMaps=param_grid,
        evaluator=evaluator,
        trainRatio=0.8,
        seed=42
    )
    with mlflow.start_run(run_name=f"{model_name}_tuned"):
        mlflow.log_param("model_name", f"{model_name}_tuned")
        mlflow.log_param("label", label)
        mlflow.log_param("tuning", "TrainValidationSplit")

        tvs_model = tvs.fit(train_df)
        best_model = tvs_model.bestModel

        preds = best_model.transform(test_df)
        metrics = eval_regression(preds, label)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        reg_stage = best_model.stages[-1]
        best_params = {p.name: reg_stage.getOrDefault(p) for p in reg_stage.params if reg_stage.isSet(p)}
        for k, v in best_params.items():
            mlflow.log_param(k, str(v))
        mlflow.spark.log_model(best_model, "model",dfs_tmpdir="/Volumes/workspace/ecommerce/ecommerce_data/mlruns/tmp", pip_requirements=["pyspark==4.0.0", "mlflow"])
    return best_model, metrics, best_params
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
rf = RandomForestRegressor(featuresCol="features", labelCol=label, seed=42)
rf_grid = (ParamGridBuilder()
           .addGrid(rf.numTrees, [50, 100])
           .addGrid(rf.maxDepth, [5, 10])
           .build())
best_rf, rf_metrics, rf_params = tune("spark_rf", rf, rf_grid)
print("Best RF:", rf_metrics, rf_params)
gbt = GBTRegressor(featuresCol="features", labelCol=label, seed=42)
gbt_grid = (ParamGridBuilder()
            .addGrid(gbt.maxDepth, [3, 5])
            .addGrid(gbt.maxIter, [30, 80])
            .addGrid(gbt.stepSize, [0.05, 0.1])
            .build())

best_gbt, gbt_metrics, gbt_params = tune("spark_gbt", gbt, gbt_grid)
print("Best GBT:", gbt_metrics, gbt_params)


In [0]:
# Feature importance (RF/GBT)
import pandas as pd

def get_importance(pipeline_model):
    reg = pipeline_model.stages[-1]
    if not hasattr(reg, "featureImportances"):
        return None
    arr = reg.featureImportances.toArray().tolist()
    return pd.DataFrame({"feature": feature_cols, "importance": arr}).sort_values("importance", ascending=False)

rf_imp = get_importance(best_rf)
gbt_imp = get_importance(best_gbt)

print("RF importance")
display(rf_imp)

print("GBT importance")
display(gbt_imp)


In [0]:
# Pick the winner (lowest RMSE)
all_scores = results + [
    {"model": "spark_rf_tuned", **rf_metrics},
    {"model": "spark_gbt_tuned", **gbt_metrics},
]

winner = sorted(all_scores, key=lambda x: x["rmse"])[0]
winner
